# Deep Past Initiative – Submit Notebook (ByT5-base)

- 対象: `[4-1]submit-notebook-v4.ipynb`
- 元: `notebooks/002/[4]submit-notebook-v1.ipynb`
- 種別: 枝分かれ（submit v1 ベースで train v4 系の前処理へ更新）
- 変更点:
  - モデル: `notebooks/002/[2]dpc-starter-train-v4.ipynb` で学習した ByT5-base（保存ディレクトリ）を読み込む前提に変更
  - 前処理/後処理: `notebooks/002/[2]dpc-starter-train-v4.ipynb` の `normalize_transliteration()` / `postprocess_translation()` と同等に統一
  - 推論: 既存の submit notebook の I/O 仕様（`test.csv` → `submission.csv`）は維持

## Model path
`MODEL_DIR` を学習済みモデル（`fulltrain_byt5-base_multitask`）が置かれたディレクトリに合わせてください。


In [1]:
import os
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from IPython.display import display
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


'false'

In [2]:
@dataclass
class CFG:
    # Competition data path (Kaggle may mount it in either location)
    input_dirs: tuple[str, ...] = (
        "/kaggle/input/competitions/deep-past-initiative-machine-translation",
        "/kaggle/input/deep-past-initiative-machine-translation",
    )

    # >>> Replace this later (intentionally centralized)
    # Example (dataset): /kaggle/input/<your-dataset>/<your-model-folder>
    # Example (model):   /kaggle/input/models/<owner>/<model>/pytorch/<variant>/<version>
    model_dir: str = os.environ.get("MODEL_DIR", "/kaggle/input/notebooks/tatsuokoshida/2-dpc-starter-train-v4/byt5-akkadian-model/fulltrain_byt5-base_multitask")

    output_dir: str = "/kaggle/working"

    # Tokenization / generation (match train notebook defaults)
    max_input_length: int = 512
    max_new_tokens: int = 512

    # Inference
    batch_size: int = 8
    num_workers: int = 2
    num_beams: int = 4
    early_stopping: bool = True

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


cfg = CFG()
print("PyTorch:", torch.__version__)
print("Device :", cfg.device)


PyTorch: 2.9.0+cu126
Device : cuda


In [3]:
PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

_SUBSCRIPT_NUM_MAP = str.maketrans({
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
})

_TRANSLATION_D1_PATTERNS = [
    (re.compile(r"\b(?:fem|sing|pl|plural)\.?(?=\s|$)", flags=re.IGNORECASE), " "),
    (re.compile(r"\(\?\)"), " "),
]

_TRANSLITERATION_REPLACEMENTS = [
    (re.compile(r"\bKÙ\.B\.\b"), "KÙ.BABBAR"),
    (re.compile(r"\bKU3\.B\.\b", flags=re.IGNORECASE), "KÙ.BABBAR"),
]

_DECIMAL_TO_FRACTION_MAP = {
    # 1/12 系
    "0.0833": "1/12",
    "0.1667": "1/6",
    "0.25": "1/4",
    "0.3333": "1/3",
    "0.33333": "1/3",
    "0.4167": "5/12",
    "0.5": "1/2",
    "0.5833": "7/12",
    "0.6667": "2/3",
    "0.66667": "2/3",
    "0.75": "3/4",
    "0.8333": "5/6",
    "0.9167": "11/12",

    # 10分率 / 5分率
    "0.1": "1/10",
    "0.2": "1/5",
    "0.3": "3/10",
    "0.4": "2/5",
    "0.6": "3/5",
    "0.7": "7/10",
    "0.8": "4/5",
    "0.9": "9/10",

    # 8分率
    "0.125": "1/8",
    "0.375": "3/8",
    "0.625": "5/8",
    "0.875": "7/8",

    # 6分率
    "0.1666": "1/6",
    "0.3334": "1/3",
    "0.6666": "2/3",
    "0.8334": "5/6",

    # 9分率 / 7分率（丸め値）
    "0.1111": "1/9",
    "0.8889": "8/9",
    "0.1429": "1/7",
    "0.2857": "2/7",
    "0.4286": "3/7",
    "0.5714": "4/7",
    "0.7143": "5/7",
    "0.8571": "6/7",
}

_ROMAN_MAP = {
    # ASCII Roman numerals
    "I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000,
    # Unicode Roman numeral letters
    "Ⅰ": 1, "Ⅱ": 2, "Ⅲ": 3, "Ⅳ": 4, "Ⅴ": 5, "Ⅵ": 6, "Ⅶ": 7, "Ⅷ": 8, "Ⅸ": 9,
    "Ⅹ": 10, "Ⅺ": 11, "Ⅻ": 12, "Ⅼ": 50, "Ⅽ": 100, "Ⅾ": 500, "Ⅿ": 1000,
}

_GAP_MARKER_PATTERNS = [
    # Transliteration 側の gap 表現を `<gap>` に寄せる（host 推奨の test 整合観点）
    (re.compile(r"\(\s*(?:large\s+)?break\s*\)", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\(\s*\d+\s+broken\s+lines?\s*\)", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"<\s*big_gap\s*>", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\bbig_gap\b", flags=re.IGNORECASE), " <gap> "),
    (re.compile(r"\[\s*[xX]+\s*\]"), " <gap> "),
    (re.compile(r"\b[xX]{1,}\b"), " <gap> "),
]


def _collapse_gaps(text: str) -> str:
    # `<gap>` の前後の空白を一旦整えてから縮約する
    text = re.sub(r"\s*<gap>\s*", " <gap> ", text)

    # `<gap>. <gap>` / `<gap>, <gap>` のように句読点を挟んだ連続も `<gap>` に縮約
    while True:
        new_text = re.sub(r"<gap>\s*[\.,]\s*<gap>", "<gap>", text)
        new_text = re.sub(r"<gap>\s+<gap>", "<gap>", new_text)
        if new_text == text:
            break
        text = new_text
    return text


def _normalize_decimal_key(s: str) -> str:
    # 末尾ゼロを削って辞書キーを正規化（0.5000 -> 0.5）
    if "." in s:
        s = s.rstrip("0").rstrip(".")
    if s.startswith("."):
        s = "0" + s
    return s


def _fraction_from_decimal_part(decimal_part: float):
    if decimal_part <= 0.0:
        return None
    for digits in (5, 4, 3):
        rounded_key = _normalize_decimal_key(f"{decimal_part:.{digits}f}")
        mapped = _DECIMAL_TO_FRACTION_MAP.get(rounded_key)
        if mapped is not None:
            return mapped
    return None


def _replace_decimals_with_fractions(text: str) -> str:
    # 任意の小数値を「整数部 + 小数部」に分け、小数部のみ分数へ寄せる
    # 例: 8.33333 -> 8 1/3
    def repl(match):
        token = match.group(0)
        key = _normalize_decimal_key(token)

        try:
            value = float(key)
        except ValueError:
            return token

        sign = "-" if value < 0 else ""
        abs_value = abs(value)
        integer_part = int(abs_value)
        decimal_part = abs_value - integer_part
        if decimal_part <= 0:
            return token

        mapped = _fraction_from_decimal_part(decimal_part)
        if mapped is None:
            return token

        if integer_part == 0:
            return sign + mapped
        return f"{sign}{integer_part} {mapped}"

    return re.sub(r"(?<![\d/])[-+]?\d+\.\d+(?!\d)", repl, text)


def _roman_to_int(token: str) -> int:
    token = token.upper()
    total = 0
    prev = 0
    for ch in reversed(token):
        value = _ROMAN_MAP.get(ch, 0)
        if value < prev:
            total -= value
        else:
            total += value
            prev = value
    return total


def _normalize_month_roman(text: str) -> str:
    def repl(match):
        roman = match.group(1)
        value = _roman_to_int(roman)
        if value <= 0:
            return match.group(0)
        return f"month {value}"

    return re.sub(r"\bmonth\s+([IVXLC]+)\b", repl, text, flags=re.IGNORECASE)


def _drop_translation_alternatives(text: str) -> str:
    # 分数 (1/12) は残し、語の代替 (you / she) は左側に寄せる
    prev = None
    pattern = re.compile(
        r"\b([A-Za-z][A-Za-z'\-]*(?:\s+[A-Za-z][A-Za-z'\-]*){0,3})\s*/\s*([A-Za-z][A-Za-z'\-]*(?:\s+[A-Za-z][A-Za-z'\-]*){0,3})\b"
    )
    while prev != text:
        prev = text
        text = pattern.sub(r"\1", text)
    return text


def normalize_transliteration(text: str) -> str:
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(_SUBSCRIPT_NUM_MAP)

    # ダッシュ類の統一
    text = re.sub(r"[‐‑–—]", "-", text)

    # 下付き `ₓ` の除去（ノイズになりやすい）
    text = text.replace("ₓ", "")
    text = text.replace("Ḫ", "H").replace("ḫ", "h")

    # 長い小数を短縮（小数点以下4桁まで）
    text = re.sub(r"(\d+\.\d{4})\d+", r"\1", text)
    text = _replace_decimals_with_fractions(text)

    for pat, repl in _TRANSLITERATION_REPLACEMENTS:
        text = pat.sub(repl, text)

    # gap 表現を `<gap>` に統一
    text = text.replace("…", " <gap> ")
    for pat, repl in _GAP_MARKER_PATTERNS:
        text = pat.sub(repl, text)

    # 決定詞の表記を test 側へ寄せる
    text = re.sub(r"\((d|ki)\)", r"{\1}", text, flags=re.IGNORECASE)
    text = re.sub(r"\(([A-ZŠṢṬḪ]+)\)", r"\1", text)

    text = _collapse_gaps(text)

    # Collapse whitespace noise introduced by normalization/replacements.
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_translation_d1(text: str) -> str:
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)

    for pattern, repl in _TRANSLATION_D1_PATTERNS:
        text = pattern.sub(repl, text)

    # `PN` は `<gap>` に寄せる（host 推奨）
    text = re.sub(r"\bPN\b", " <gap> ", text)
    text = re.sub(r"-gold\b", "pašallum gold", text)
    text = re.sub(r"-tax\b", "šadduātum tax", text)
    text = re.sub(r"-textiles\b", "kutānum textiles", text)

    text = _drop_translation_alternatives(text)
    text = _normalize_month_roman(text)
    text = _replace_decimals_with_fractions(text)

    text = _collapse_gaps(text)

    text = re.sub(r"\s+", " ", text).strip()
    return text


def postprocess_translation(text: str) -> str:
    # 生成後の後処理（train v4 と整合）
    normalized = normalize_translation_d1(text)
    # 提出フォーマットの empty values 回避
    if normalized == "":
        return "<gap>"
    return normalized


def resolve_input_dir(input_dirs: tuple[str, ...]) -> str:
    for d in input_dirs:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Competition input directory not found. Tried: " + ", ".join(input_dirs)
    )


def build_inputs(test_df: pd.DataFrame) -> tuple[list[str], list[str]]:
    if "id" not in test_df.columns:
        raise ValueError(f"test.csv must contain 'id'. columns={list(test_df.columns)}")
    ids = test_df["id"].tolist()

    if "task" in test_df.columns:
        tasks = test_df["task"].astype(str).tolist()
    else:
        tasks = ["akk2en"] * len(test_df)

    if "source" in test_df.columns:
        sources = test_df["source"].astype(str).tolist()
    elif "transliteration" in test_df.columns:
        sources = test_df["transliteration"].astype(str).tolist()
    else:
        raise ValueError(
            "test.csv must contain either 'source' or 'transliteration'. "
            f"columns={list(test_df.columns)}"
        )

    inputs: list[str] = []
    for t, s in zip(tasks, sources):
        if t == "akk2en":
            inputs.append(PREFIX_AKK_EN + normalize_transliteration(s))
        elif t == "en2akk":
            inputs.append(PREFIX_EN_AKK + normalize_translation_d1(s))
        else:
            raise ValueError(f"Unknown task: {t}")

    return ids, inputs


In [4]:
def resolve_model_dir(model_dir: str) -> str:
    candidates = [
        model_dir,
        # Useful when you ran training in the same notebook session and kept outputs in /kaggle/working
        "/kaggle/working/byt5-akkadian-model/fulltrain_byt5-base_multitask",
    ]
    for d in candidates:
        if Path(d).exists():
            return d
    raise FileNotFoundError(
        "Model directory not found. Set MODEL_DIR to a mounted path containing the saved model. "
        f"Tried: {candidates}"
    )


input_dir = resolve_input_dir(cfg.input_dirs)
test_path = f"{input_dir}/test.csv"

print("Test:", test_path)

test_df = pd.read_csv(test_path)
print("Test rows:", len(test_df))
print("Columns:", list(test_df.columns))

model_dir = resolve_model_dir(cfg.model_dir)
print("Model:", model_dir)

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to(cfg.device)
model.eval()

ids, inputs = build_inputs(test_df)
print("Prepared inputs:", len(inputs))


Test: /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
Test rows: 4
Columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
Model: /kaggle/input/notebooks/tatsuokoshida/2-dpc-starter-train-v4/byt5-akkadian-model/fulltrain_byt5-base_multitask


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Prepared inputs: 4


In [5]:
class TextDataset(Dataset):
    def __init__(self, texts: list[str]):
        self.texts = texts

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> str:
        return self.texts[idx]


def collate_fn(batch_texts: list[str]):
    return tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=cfg.max_input_length,
    )


ds = TextDataset(inputs)
dl = DataLoader(
    ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=(cfg.device.type == "cuda"),
)


def _move_to_device(batch: dict, device: torch.device) -> dict:
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


def postprocess_text(text: str) -> str:
    return postprocess_translation(text)


preds: list[str] = []
with torch.inference_mode():
    for batch in tqdm(dl, desc="Generating"):
        batch = _move_to_device(batch, cfg.device)
        gen_ids = model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch.get("attention_mask"),
            num_beams=cfg.num_beams,
            early_stopping=cfg.early_stopping,
            max_new_tokens=cfg.max_new_tokens,
        )
        out_texts = tokenizer.batch_decode(
            gen_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        preds.extend([postprocess_text(t) for t in out_texts])

assert len(preds) == len(ids)
print("Preds:", len(preds))
print("Example:", preds[0][:200] if preds else "<empty>")


Generating:   0%|          | 0/1 [00:00<?, ?it/s]

Preds: 4
Example: Thus Kanesh colony, say to the food of <gap> , our messenger, every single, and the colony: The tablet has come from the City.


In [6]:
sub_df = pd.DataFrame({"id": ids, "translation": preds})
sub_df["translation"] = sub_df["translation"].fillna("").astype(str)
sub_df.loc[sub_df["translation"].str.strip() == "", "translation"] = "<gap>"
sub_path = Path(cfg.output_dir) / "submission.csv"
sub_df.to_csv(sub_path, index=False)

print("Submission preview:")
display(sub_df.head())
print("Saved to:", str(sub_path))


Submission preview:


,id,translation
0,0,"Thus Kanesh colony, say to the food of <gap> ,..."
1,1,When the tablet comes up from the City when we...
2,2,"Since you hear our letter, either he gave it t..."
3,3,I send our tablets to every single colony and ...


Saved to: /kaggle/working/submission.csv
